# Pipeline de recertification KYC — Qwen2.5-VL uniquement (exécution séquentielle)

Ce notebook extrait les informations de **JUSTIFICATIF IDENTITE.PDF** et
**JUSTIFICATIF DOMICILE.PDF** pour chaque client (archive ZIP, dossiers nommés par l'ID
client), puis compare le résultat avec `tiers.csv`.

**Règles appliquées suite à vos retours :**
- **Un seul modèle : Qwen2.5-VL**, chargé avec `AutoModelForVision2Seq` (aucune autre classe,
  aucun fallback en cascade).
- **Aucune fonction géante exécutée uniquement en fin de notebook** : chaque cellule définit
  une petite brique et l'exécute/teste immédiatement sur des données réelles, pour repérer les
  erreurs cellule par cellule.
- **Comparaison avec `tiers.csv`** sur la clé `ID TIERS`, avec les colonnes : nom
  patronymique, prénom patronymique, numéro d'identification national (PP), lieu de
  naissance, date de naissance, date d'expiration du document → export d'un fichier de
  résultat MATCH / NO MATCH par champ.

> ⚠️ Vos deux colonnes citées "date expiration du document" semblent être un doublon dans
> l'énoncé — je n'utilise qu'une seule colonne `date_expiration_document`. Dites-moi si vous
> vouliez en réalité deux colonnes différentes (ex. date de délivrance + date d'expiration),
> j'ajusterai le mapping.


## 0. Pin de la version `transformers` (nécessaire pour `AutoModelForVision2Seq`)

`AutoModelForVision2Seq` a été **retirée dans `transformers` v5.0** (remplacée par
`AutoModelForImageTextToText`). Si votre environnement a installé la v5, l'import échoue
quel que soit le code utilisé autour. On épingle donc une version compatible qui possède à la
fois `AutoModelForVision2Seq` **et** le support de Qwen2.5-VL.

**⚠️ Après l'exécution de cette cellule, redémarrez le kernel** (Kernel → Restart), puis
reprenez l'exécution à partir de la cellule suivante.

In [ ]:
%pip install -q "transformers>=4.49,<5.0" accelerate qwen-vl-utils
print("Installation terminée. Redémarrez le kernel avant de continuer (Kernel > Restart Kernel).")


## 1. Imports

In [ ]:
import os
import re
import io
import json
import time
import zipfile
import shutil
import unicodedata
from pathlib import Path
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
from PIL import Image

import torch
from transformers import AutoProcessor, AutoModelForVision2Seq

from rapidfuzz import fuzz

try:
    import pytesseract
    HAS_TESSERACT = True
except ImportError:
    HAS_TESSERACT = False

print("Torch CUDA disponible :", torch.cuda.is_available())


## 2. Configuration

In [ ]:
MODEL_PATH = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen2.5-VL-7B-instruct/main"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ZIP_PATH = "/mnt/data/dossiers_clients.zip"
WORKDIR = Path("/tmp/kyc_extraction")
TIERS_CSV_PATH = "/mnt/data/tiers.csv"
OUTPUT_DIR = Path("/mnt/data/output_kyc")

WORKDIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_DOCS = {
    "identite": "JUSTIFICATIF IDENTITE",
    "domicile": "JUSTIFICATIF DOMICILE",
}

DPI = 300
FUZZY_FILENAME_THRESHOLD = 80
MATCH_THRESHOLD_TIERS = 85.0   # seuil de similarité pour considérer 2 valeurs comme identiques

# Mapping colonnes tiers.csv -> noms normalisés utilisés dans ce notebook
TIERS_COLUMN_MAPPING = {
    "ID TIERS": "id_tiers",
    "NOM PATRONYMIQUE": "nom_patronymique",
    "PRENOM PATRONYMIQUE": "prenom_patronymique",
    "NUMERO D'IDENTIFICATION NATIONAL (PP)": "numero_identification_national",
    "LIEU DE NAISSANCE": "lieu_naissance",
    "DATE DE NAISSANCE": "date_naissance",
    "DATE EXPIRATION DU DOCUMENT": "date_expiration_document",
}
print("Configuration chargée.")


## 3. Chargement du modèle Qwen2.5-VL (`AutoModelForVision2Seq`)

In [ ]:
t0 = time.time()

processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16,
    trust_remote_code=True, low_cpu_mem_usage=True,
)
model.eval().to(DEVICE)

print(f'✅ Modèle chargé en {time.time()-t0:.1f}s')


## 4. Fonction d'appel au modèle — testée immédiatement

In [ ]:
def call_qwen(image: Image.Image, prompt: str, max_new_tokens: int = 1024,
              temperature: float = 0.0) -> str:
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    text_prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    try:
        from qwen_vl_utils import process_vision_info
        image_inputs, video_inputs = process_vision_info(messages)
    except ImportError:
        image_inputs, video_inputs = [image], None

    inputs = processor(
        text=[text_prompt], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=max(temperature, 1e-5),
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]
    return output_text.strip()


In [ ]:
# Test immédiat de call_qwen sur une image factice
_test_img = Image.new("RGB", (300, 100), color="white")
_test_answer = call_qwen(_test_img, "Décris cette image en un mot.")
print("Réponse du modèle :", _test_answer)


## 5. Fonctions utilitaires : normalisation de texte et similarité floue

In [ ]:
def normalize_text(s) -> str:
    if s is None:
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[_\-]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip().upper()
    return s


def fuzzy_score(a: str, b: str) -> int:
    return fuzz.partial_ratio(normalize_text(a), normalize_text(b))


print(normalize_text("Ahmed  Ben-Ali "), "|", fuzzy_score("Justificatif Identité", "JUSTIFICATIF IDENTITE.pdf"))


## 6. Dézippage de l'archive et index des dossiers clients

In [ ]:
def extract_zip(zip_path: str, workdir: Path) -> Path:
    extract_to = workdir / "extracted"
    if extract_to.exists():
        shutil.rmtree(extract_to)
    extract_to.mkdir(parents=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_to)
    return extract_to


extract_root = extract_zip(ZIP_PATH, WORKDIR)
print("Archive extraite dans :", extract_root)


In [ ]:
def list_client_folders(extract_root: Path):
    folders = [p for p in extract_root.iterdir() if p.is_dir()]
    if not folders:
        folders = [p for p in extract_root.rglob("*") if p.is_dir()]
    return folders


client_folders = list_client_folders(extract_root)
print(f"{len(client_folders)} dossier(s) client trouvé(s).")
[f.name for f in client_folders[:10]]


In [ ]:
def find_target_pdfs(client_folder: Path) -> dict:
    result = {"identite": None, "domicile": None}
    pdf_files = list(client_folder.rglob("*.pdf")) + list(client_folder.rglob("*.PDF"))
    for key, target_name in TARGET_DOCS.items():
        best_path, best_score = None, 0
        for pdf in pdf_files:
            score = fuzzy_score(pdf.stem, target_name)
            if score > best_score:
                best_score, best_path = score, pdf
        if best_score >= FUZZY_FILENAME_THRESHOLD:
            result[key] = best_path
    return result


rows = []
for folder in client_folders:
    targets = find_target_pdfs(folder)
    rows.append({
        "client_id": folder.name,
        "path_identite": str(targets["identite"]) if targets["identite"] else None,
        "path_domicile": str(targets["domicile"]) if targets["domicile"] else None,
    })
client_index = pd.DataFrame(rows)

missing = client_index[client_index["path_identite"].isna() | client_index["path_domicile"].isna()]
if len(missing):
    print(f"⚠️  {len(missing)} client(s) avec au moins un document cible manquant.")
client_index.head()


## 7. PDF → image : test sur le premier client complet

In [ ]:
import fitz  # PyMuPDF

def pdf_to_images(pdf_path: str, dpi: int = DPI):
    images = []
    zoom = dpi / 72.0
    mat = fitz.Matrix(zoom, zoom)
    with fitz.open(pdf_path) as doc:
        for page in doc:
            pix = page.get_pixmap(matrix=mat, alpha=False)
            img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
            images.append(img)
    return images


def pick_main_page(images):
    if len(images) == 1:
        return images[0]
    scored = [(np.array(img.convert("L")).std(), img) for img in images]
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1]


sample_row = client_index.dropna(subset=["path_identite", "path_domicile"]).iloc[0]
sample_client_id = sample_row["client_id"]
print("Client de test :", sample_client_id)

sample_identite_img = pick_main_page(pdf_to_images(sample_row["path_identite"]))
sample_domicile_img = pick_main_page(pdf_to_images(sample_row["path_domicile"]))
print("Taille image identité :", sample_identite_img.size)
print("Taille image domicile :", sample_domicile_img.size)
sample_identite_img.resize((400, int(400 * sample_identite_img.height / sample_identite_img.width)))


## 8. Prétraitement (rotation, deskew, contraste) — appliqué au cas de test

In [ ]:
def deskew(image: Image.Image) -> Image.Image:
    arr = np.array(image.convert("L"))
    arr = cv2.GaussianBlur(arr, (5, 5), 0)
    _, thresh = cv2.threshold(arr, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    coords = np.column_stack(np.where(thresh > 0))
    if len(coords) < 50:
        return image
    angle = cv2.minAreaRect(coords)[-1]
    angle = -(90 + angle) if angle < -45 else -angle
    if abs(angle) < 0.5 or abs(angle) > 20:
        return image
    (h, w) = arr.shape
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    rotated = cv2.warpAffine(np.array(image), M, (w, h),
                              flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    return Image.fromarray(rotated)


def enhance_for_ocr(image: Image.Image) -> Image.Image:
    arr = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    arr = clahe.apply(arr)
    arr = cv2.fastNlMeansDenoising(arr, h=10)
    return Image.fromarray(cv2.cvtColor(arr, cv2.COLOR_GRAY2RGB))


def detect_rotation_tesseract(image: Image.Image):
    if not HAS_TESSERACT:
        return None
    try:
        osd = pytesseract.image_to_osd(image)
        m = re.search(r"Rotate: (\d+)", osd)
        conf = re.search(r"Orientation confidence: ([\d.]+)", osd)
        if m and conf and float(conf.group(1)) >= 1.0:
            return int(m.group(1))
    except Exception:
        return None
    return None


def rotate_image(image: Image.Image, angle: int) -> Image.Image:
    if angle % 360 == 0:
        return image
    return image.rotate(-angle, expand=True)


def detect_rotation_qwen(image: Image.Image) -> int:
    prompt = (
        "Regarde ce document scanné. Indique uniquement l'angle de rotation horaire "
        "nécessaire pour que le texte soit parfaitement à l'endroit. Réponds strictement "
        "par un seul nombre parmi 0, 90, 180, 270."
    )
    answer = call_qwen(image, prompt)
    match = re.search(r"\b(0|90|180|270)\b", answer)
    return int(match.group(1)) if match else 0


In [ ]:
angle_identite = detect_rotation_tesseract(sample_identite_img)
if angle_identite is None:
    angle_identite = detect_rotation_qwen(sample_identite_img)
print("Angle détecté (identité) :", angle_identite)

sample_identite_clean = rotate_image(sample_identite_img, angle_identite)
sample_identite_clean = deskew(sample_identite_clean)
sample_identite_clean = enhance_for_ocr(sample_identite_clean)

angle_domicile = detect_rotation_tesseract(sample_domicile_img)
if angle_domicile is None:
    angle_domicile = detect_rotation_qwen(sample_domicile_img)
print("Angle détecté (domicile) :", angle_domicile)

sample_domicile_clean = rotate_image(sample_domicile_img, angle_domicile)
sample_domicile_clean = deskew(sample_domicile_clean)
sample_domicile_clean = enhance_for_ocr(sample_domicile_clean)

sample_identite_clean.resize((400, int(400 * sample_identite_clean.height / sample_identite_clean.width)))


## 9. Détection du pays d'émission — testée sur le cas réel

In [ ]:
DZ_KEYWORDS = [
    "REPUBLIQUE ALGERIENNE", "REPUBLIQUE ALGERIENNE DEMOCRATIQUE ET POPULAIRE",
    "الجمهورية الجزائرية", "وزارة الداخلية", "BLIDA", "ALGER", "ORAN", "CONSTANTINE",
    "WILAYA", "COMMUNE DE", "الجزائر", "بطاقة التعريف الوطنية", "CARTE NATIONALE D'IDENTITE",
]

def heuristic_country_score(raw_text: str) -> float:
    text_norm = normalize_text(raw_text)
    hits = sum(1 for kw in DZ_KEYWORDS if normalize_text(kw) in text_norm)
    return min(hits / 2.0, 1.0)


def detect_country(image: Image.Image) -> str:
    prompt = (
        "Ce document est-il une pièce d'identité ou un justificatif de domicile émis en "
        "Algérie ? Réponds strictement par 'DZ' si oui, ou 'OTHER' si le document provient "
        "d'un autre pays."
    )
    answer = call_qwen(image, prompt)
    return "DZ" if "DZ" in answer.upper() else "OTHER"


sample_country = detect_country(sample_identite_clean)
print("Pays détecté pour le client de test :", sample_country)


## 10. Schémas d'extraction (documents émis en Algérie)

In [ ]:
SCHEMA_DZ_IDENTITE = {
    "type_document": "carte_identite_nationale | passeport | permis_conduire",
    "nom": "string (nom de famille, اللقب)",
    "prenom": "string (prénom, الاسم)",
    "date_naissance": "YYYY-MM-DD",
    "lieu_naissance": "string",
    "sexe": "M | F",
    "numero_piece": "string (numéro de la carte / passeport)",
    "date_delivrance": "YYYY-MM-DD",
    "date_expiration": "YYYY-MM-DD",
    "autorite_delivrance": "string (wilaya / commune / daira)",
    "adresse": "string (adresse figurant sur la pièce, si présente)",
}

SCHEMA_DZ_DOMICILE = {
    "type_document": "facture_sonelgaz | facture_ADE | attestation_communale | autre",
    "nom_titulaire": "string",
    "prenom_titulaire": "string",
    "adresse_complete": "string",
    "commune": "string",
    "wilaya": "string",
    "code_postal": "string ou null",
    "date_document": "YYYY-MM-DD",
    "numero_reference": "string ou null (n° facture / n° acte)",
}

def build_schema_prompt(schema: dict, doc_label: str) -> str:
    schema_json = json.dumps(schema, ensure_ascii=False, indent=2)
    return f\"\"\"Tu es un expert en extraction de données KYC. Le document ci-joint est un
{doc_label} émis en Algérie. Le texte peut être en arabe, en français, ou bilingue.

Extrait les informations selon EXACTEMENT le schéma JSON suivant (mêmes clés). Si une
information est absente ou illisible, mets la valeur null. Ne complète jamais un champ par
une supposition. Normalise les dates au format YYYY-MM-DD.

Schéma attendu :
{schema_json}

Réponds UNIQUEMENT avec un objet JSON valide, sans texte avant ni après, sans balises
markdown.\"\"\"


def build_generic_prompt(doc_label: str) -> str:
    return f\"\"\"Tu es un expert en extraction de données KYC. Le document ci-joint est un
{doc_label} émis dans un pays autre que l'Algérie (format inconnu à l'avance).

Identifie le pays d'émission et le type précis de document, puis extrait toutes les
informations pertinentes que tu peux lire avec certitude (nom, prénom, date de naissance,
numéro de pièce, dates de délivrance/expiration, adresse complète, autorité émettrice, etc.).

Réponds UNIQUEMENT avec un objet JSON de la forme :
{{
  "pays_detecte": "...",
  "type_document": "...",
  "champs": {{ "nom_du_champ": "valeur", ... }}
}}

N'invente aucune valeur. Normalise les dates au format YYYY-MM-DD quand c'est possible. Pas
de texte hors du JSON, pas de markdown.\"\"\"

print("Schémas et prompts définis.")


## 11. Parsing JSON robuste — testé immédiatement

In [ ]:
def parse_json_safe(raw_output: str):
    cleaned = raw_output.strip()
    cleaned = re.sub(r"^```json\s*|\s*```$", "", cleaned, flags=re.MULTILINE)
    cleaned = re.sub(r"^```\s*|\s*```$", "", cleaned, flags=re.MULTILINE)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass
    return None


print(parse_json_safe('Voici le résultat : ```json\n{"nom": "TEST"}\n``` merci'))


## 12. Extraction sur le document identité du client de test

In [ ]:
doc_label = "carte d'identité / pièce d'identité"
if sample_country == "DZ":
    prompt_identite = build_schema_prompt(SCHEMA_DZ_IDENTITE, doc_label)
else:
    prompt_identite = build_generic_prompt(doc_label)

raw_output_identite = call_qwen(sample_identite_clean, prompt_identite)
data_identite = parse_json_safe(raw_output_identite)

if data_identite is None:
    print("⚠️ JSON invalide, nouvel essai...")
    raw_output_identite = call_qwen(sample_identite_clean, prompt_identite)
    data_identite = parse_json_safe(raw_output_identite)

print(json.dumps(data_identite, ensure_ascii=False, indent=2))


## 13. Extraction sur le document domicile du client de test

In [ ]:
doc_label = "justificatif de domicile"
if sample_country == "DZ":
    prompt_domicile = build_schema_prompt(SCHEMA_DZ_DOMICILE, doc_label)
else:
    prompt_domicile = build_generic_prompt(doc_label)

raw_output_domicile = call_qwen(sample_domicile_clean, prompt_domicile)
data_domicile = parse_json_safe(raw_output_domicile)

if data_domicile is None:
    print("⚠️ JSON invalide, nouvel essai...")
    raw_output_domicile = call_qwen(sample_domicile_clean, prompt_domicile)
    data_domicile = parse_json_safe(raw_output_domicile)

print(json.dumps(data_domicile, ensure_ascii=False, indent=2))


## 14. Boucle sur tous les clients

Le pipeline a été validé étape par étape ci-dessus sur un cas réel. On l'applique maintenant
à tous les clients de `client_index`, en réutilisant telles quelles les fonctions déjà
testées (pas de nouvelle fonction géante : juste une boucle explicite, visible et modifiable
cellule par cellule).

In [ ]:
def extract_one_side(image: Image.Image, doc_type: str, country: str) -> dict:
    doc_label = "carte d'identité / pièce d'identité" if doc_type == "identite" else \
                "justificatif de domicile"
    schema = SCHEMA_DZ_IDENTITE if doc_type == "identite" else SCHEMA_DZ_DOMICILE
    prompt = build_schema_prompt(schema, doc_label) if country == "DZ" else build_generic_prompt(doc_label)

    data, erreur = None, None
    for attempt in range(2):
        raw_output = call_qwen(image, prompt)
        data = parse_json_safe(raw_output)
        if data is not None:
            break
        erreur = f"JSON invalide (tentative {attempt+1}) : {raw_output[:200]}"
    return {"data": data, "erreur": erreur}


In [ ]:
results = []
checkpoint_path = OUTPUT_DIR / "extraction_checkpoint.jsonl"
if checkpoint_path.exists():
    checkpoint_path.unlink()

for i, row in client_index.iterrows():
    client_id = row["client_id"]
    entry = {"client_id": client_id}

    for doc_type, path_col in [("identite", "path_identite"), ("domicile", "path_domicile")]:
        pdf_path = row[path_col]
        if not pdf_path:
            entry[f"{doc_type}_ok"] = False
            entry[f"{doc_type}_pays"] = None
            continue
        img = pick_main_page(pdf_to_images(pdf_path))
        angle = detect_rotation_tesseract(img)
        if angle is None:
            angle = detect_rotation_qwen(img)
        clean_img = enhance_for_ocr(deskew(rotate_image(img, angle)))
        country = detect_country(clean_img)
        extraction = extract_one_side(clean_img, doc_type, country)

        entry[f"{doc_type}_ok"] = extraction["data"] is not None
        entry[f"{doc_type}_pays"] = country
        data = extraction["data"] or {}
        flat = {**{k: v for k, v in data.items() if k != "champs"}, **data.get("champs", {})} \
            if "champs" in data else data
        for k, v in flat.items():
            entry[f"{doc_type}__{k}"] = v

    results.append(entry)

    with open(checkpoint_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False, default=str) + "\n")

    if (i + 1) % 10 == 0:
        print(f"  ... {i+1}/{len(client_index)} clients traités")

extraction_df = pd.DataFrame(results)
extraction_df.to_csv(OUTPUT_DIR / "extraction_resultats.csv", index=False)
print(f"✅ Extraction terminée : {len(extraction_df)} clients.")
extraction_df.head()


## 15. Chargement de `tiers.csv` et normalisation des colonnes

In [ ]:
tiers_raw = pd.read_csv(TIERS_CSV_PATH, dtype=str)

# Normalisation des noms de colonnes réels vers le mapping attendu (tolère variations de
# casse/accents/espaces dans les en-têtes du fichier fourni)
col_lookup = {normalize_text(c): c for c in tiers_raw.columns}
rename_dict = {}
for expected_label, target_name in TIERS_COLUMN_MAPPING.items():
    key = normalize_text(expected_label)
    if key in col_lookup:
        rename_dict[col_lookup[key]] = target_name
    else:
        print(f"⚠️ Colonne non trouvée dans tiers.csv : {expected_label}")

tiers_df = tiers_raw.rename(columns=rename_dict)
print(f"{len(tiers_df)} tiers chargés.")
tiers_df.head()


## 16. Jointure extraction ↔ tiers.csv sur `client_id` / `id_tiers`

In [ ]:
merged_df = extraction_df.merge(
    tiers_df, left_on="client_id", right_on="id_tiers", how="left"
)
print(f"{merged_df['id_tiers'].isna().sum()} client(s) sans correspondance dans tiers.csv.")
merged_df.head()


## 17. Comparaison champ à champ (MATCH / NO MATCH)

Mapping entre les champs extraits (document identité) et les colonnes `tiers.csv`.

In [ ]:
FIELD_MAPPING = {
    "identite__nom": "nom_patronymique",
    "identite__prenom": "prenom_patronymique",
    "identite__numero_piece": "numero_identification_national",
    "identite__lieu_naissance": "lieu_naissance",
    "identite__date_naissance": "date_naissance",
    "identite__date_expiration": "date_expiration_document",
}

def norm_compare(a, b) -> float:
    a, b = normalize_text(a), normalize_text(b)
    if not a or not b:
        return 0.0
    return fuzz.token_sort_ratio(a, b)


comparison_rows = []
for _, row in merged_df.iterrows():
    for extr_col, tiers_col in FIELD_MAPPING.items():
        valeur_extraite = row.get(extr_col)
        valeur_tiers = row.get(tiers_col)
        score = norm_compare(valeur_extraite, valeur_tiers)
        statut = "MATCH" if score >= MATCH_THRESHOLD_TIERS else "NO MATCH"
        comparison_rows.append({
            "client_id": row["client_id"],
            "champ": extr_col.replace("identite__", ""),
            "valeur_extraite": valeur_extraite,
            "valeur_tiers": valeur_tiers,
            "score_similarite": round(score, 1),
            "statut": statut,
        })

comparison_df = pd.DataFrame(comparison_rows)
print(f"{(comparison_df['statut'] == 'NO MATCH').sum()} incohérence(s) détectée(s) sur {len(comparison_df)} comparaisons.")
comparison_df.head(20)


## 18. Export du fichier de comparaison final

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
xlsx_path = OUTPUT_DIR / f"comparaison_tiers_{timestamp}.xlsx"

# Vue pivot : une ligne par client, une colonne par champ, avec le statut
pivot_statut = comparison_df.pivot(index="client_id", columns="champ", values="statut")
pivot_statut["nb_no_match"] = (pivot_statut == "NO MATCH").sum(axis=1)
pivot_statut = pivot_statut.sort_values("nb_no_match", ascending=False)

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    extraction_df.to_excel(writer, sheet_name="extraction_brute", index=False)
    comparison_df.to_excel(writer, sheet_name="detail_comparaison", index=False)
    pivot_statut.to_excel(writer, sheet_name="synthese_par_client")

print(f"✅ Fichier de comparaison exporté : {xlsx_path}")
pivot_statut.head(20)


## 19. Notes

- **`AutoModelForVision2Seq`** exige `transformers < 5.0` (section 0). Si votre environnement
  ne peut pas être modifié (image Docker partagée, politique IT), la seule alternative
  fonctionnelle reste `AutoModelForImageTextToText` — la classe a été supprimée, pas
  simplement renommée en usage, donc aucun contournement de code ne peut faire fonctionner
  `AutoModelForVision2Seq` sous `transformers>=5.0`.
- **Colonnes `tiers.csv`** : vérifiez `TIERS_COLUMN_MAPPING` (section 2) contre les en-têtes
  réels de votre fichier si des colonnes sont signalées "non trouvées" à la section 15.
- **Seuils** (`FUZZY_FILENAME_THRESHOLD`, `MATCH_THRESHOLD_TIERS`) à calibrer sur un
  échantillon réel avant mise en production.
- **Revue humaine** : le fichier `comparaison_tiers_*.xlsx` (onglet `synthese_par_client`,
  trié par `nb_no_match` décroissant) sert de file de contrôle manuel, pas de rejet
  automatique.
